In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import numpy as np
import cv2
from ultralytics import YOLO

# -------------------------
# 1. CHECK GPU
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 2. DATASET PIPELINES
# -------------------------
face_train_dir = r"D:\RuhunaNew\Academic\Research\Facial_Recog_Repo\Group_50_Repo\Multi_Model\data\faces\train"
face_val_dir = r"D:\RuhunaNew\Academic\Research\Facial_Recog_Repo\Group_50_Repo\Multi_Model\data\faces\val"
hand_train_dir = r"D:\RuhunaNew\Academic\Research\Facial_Recog_Repo\Group_50_Repo\Multi_Model\data\hands\train"
hand_val_dir = r"D:\RuhunaNew\Academic\Research\Facial_Recog_Repo\Group_50_Repo\Multi_Model\data\hands\val"

face_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(128, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

hand_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(128, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

face_train_ds = ImageFolder(face_train_dir, transform=face_transform)
face_val_ds = ImageFolder(face_val_dir, transform=face_transform)
hand_train_ds = ImageFolder(hand_train_dir, transform=hand_transform)
hand_val_ds = ImageFolder(hand_val_dir, transform=hand_transform)

face_train_loader = DataLoader(face_train_ds, batch_size=32, shuffle=True)
face_val_loader = DataLoader(face_val_ds, batch_size=32, shuffle=False)
hand_train_loader = DataLoader(hand_train_ds, batch_size=16, shuffle=True)
hand_val_loader = DataLoader(hand_val_ds, batch_size=16, shuffle=False)

# -------------------------
# 3. CUSTOM MODEL with YOLOv11n backbone
# -------------------------

class YOLO11nFeatureExtractor(nn.Module):
    def _init_(self, feature_dim=256):
        super()._init_()
        yolo = YOLO("yolo11n.pt")
        # Select first 6 layers as backbone - adjust if needed after inspecting model
        self.backbone = nn.Sequential(*list(yolo.model.model.children())[:6])
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Use a dummy input to determine feature size dynamically
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 640, 640)
            feats = self.backbone(dummy_input)
            pooled = self.pool(feats)
            flattened_size = pooled.view(1, -1).shape[1]
        
        self.fc = nn.Linear(flattened_size, feature_dim)
    
    def forward(self, x):
        features = self.backbone(x)
        pooled = self.pool(features).flatten(1)
        out = self.fc(pooled)
        return out


class MultiTaskModel(nn.Module):
    def _init_(self, feature_dim=256, face_classes=9, hand_classes=3):
        super()._init_()
        self.feature_extractor = YOLO11nFeatureExtractor(feature_dim=feature_dim)
        self.face_head = nn.Linear(feature_dim, face_classes)
        self.hand_head = nn.Linear(feature_dim, hand_classes)
    
    def forward(self, x):
        features = self.feature_extractor(x)
        face_outputs = torch.softmax(self.face_head(features), dim=1)
        hand_outputs = torch.softmax(self.hand_head(features), dim=1)
        return face_outputs, hand_outputs

model = MultiTaskModel().to(device)

# -------------------------
# 4. TRAINING LOOP with head-specific early stopping
# -------------------------

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

face_patience = 5
hand_patience = 5
face_best_acc = 0
hand_best_acc = 0
face_wait = 0
hand_wait = 0
face_frozen = False
hand_frozen = False
epochs = 50

for epoch in range(epochs):
    model.train()
    steps = min(len(face_train_loader), len(hand_train_loader))
    pbar = tqdm(total=steps, desc=f"Epoch {epoch+1}/{epochs}")
    
    face_iter = iter(face_train_loader)
    hand_iter = iter(hand_train_loader)
    
    for _ in range(steps):
        # Face batch
        face_imgs, face_labels = next(face_iter)
        face_imgs, face_labels = face_imgs.to(device), face_labels.to(device)
        
        # Hand batch
        hand_imgs, hand_labels = next(hand_iter)
        hand_imgs, hand_labels = hand_imgs.to(device), hand_labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward face, freeze if needed
        if not face_frozen:
            face_out, _ = model(face_imgs)
            face_loss = criterion(face_out, face_labels)
        else:
            face_loss = 0
        
        # Forward hand, freeze if needed
        if not hand_frozen:
            _, hand_out = model(hand_imgs)
            hand_loss = criterion(hand_out, hand_labels)
        else:
            hand_loss = 0
        
        loss = 0
        if not face_frozen:
            loss += face_loss
        if not hand_frozen:
            loss += hand_loss
        
        loss.backward()
        optimizer.step()
        
        pbar.update(1)
    pbar.close()
    
    # Validation function
    def validate(loader, head='face'):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(device), labels.to(device)
                face_out, hand_out = model(imgs)
                preds = face_out if head=='face' else hand_out
                predicted = torch.argmax(preds, dim=1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        return correct / total
    
    face_acc = validate(face_val_loader, head='face')
    hand_acc = validate(hand_val_loader, head='hand')
    
    print(f"Epoch {epoch+1}: Face Val Acc = {face_acc:.3f}, Hand Val Acc = {hand_acc:.3f}")
    
    if not face_frozen:
        if face_acc > face_best_acc:
            face_best_acc = face_acc
            face_wait = 0
        else:
            face_wait += 1
            if face_wait >= face_patience:
                print("⏸️ Freezing face head (early stopped)")
                face_frozen = True
    
    if not hand_frozen:
        if hand_acc > hand_best_acc:
            hand_best_acc = hand_acc
            hand_wait = 0
        else:
            hand_wait += 1
            if hand_wait >= hand_patience:
                print("⏸️ Freezing hand head (early stopped)")
                hand_frozen = True
    
    if face_frozen and hand_frozen:
        print("✅ Both heads converged. Stopping training.")
        break

print("✅ Training complete.")

# -------------------------
# 5. INFERENCE PIPELINE
# -------------------------

def predict_emotion(img_path, face_detector, model, threshold=0.6):
    model.eval()
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.3, 5)
    img_resized = cv2.resize(img, (128, 128))
    img_tensor = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)
    
    with torch.no_grad():
        face_preds, hand_preds = model(img_tensor)
    
    face_prob, hand_prob = torch.max(face_preds, 1), torch.max(hand_preds, 1)
    
    if len(faces) > 0 and face_prob.values.item() > threshold:
        return ("face", face_prob.indices.item(), face_prob.values.item())
    else:
        return ("hand", hand_prob.indices.item(), hand_prob.values.item())